In [2]:
# 강화학습은 정답(라벨) 없이 에이전트가 직접 행동하고 보상을 받아 학습하는 방식이다.
# 현재 상태 → 행동 선택 → 보상 확인 → 다음 행동 개선

In [3]:
# GridWorld 환경 설정
import numpy as np
import random

np.random.seed(42)
random.seed(42)

# 3행 4열 격자 세계
ROWS = 3
COLS = 4

START = (0, 0)  # 시작 위치
GOAL  = (2, 3)  # 목표 (도달 시 +10, 에피소드 종료)
TRAP  = (1, 1)  # 함정 (도달 시 -10, 에피소드 종료)

# 행동 정의: 방향별 (행 변화량, 열 변화량)
actions = {
    0: (-1,  0),  # 위
    1: ( 1,  0),  # 아래
    2: ( 0, -1),  # 왼쪽
    3: ( 0,  1),  # 오른쪽
}

action_names = {0: '위', 1: '아래', 2: '왼쪽', 3: '오른쪽'}

num_states  = ROWS * COLS     # 12개 상태
num_actions = len(actions)    # 4개 행동

# Q-table 초기화: 모든 값을 0으로 시작
# shape: (12, 4) → 12개 상태 × 4개 행동
Q = np.zeros((num_states, num_actions))

In [4]:
# 위치 ↔ 상태 번호 변환

# 위치 (row, col) → 상태 번호 (0~11)
# 예: (2, 3) → 2 * 4 + 3 = 11
def pos_to_state(pos):
    row, col = pos
    return row * COLS + col

# 상태 번호 → 위치 (row, col)
# 예: 11 → (11 // 4, 11 % 4) = (2, 3)
def state_to_pos(state):
    row = state // COLS
    col = state % COLS
    return (row, col)

In [5]:
# 하이퍼 파라미터

alpha         = 0.1    # 학습률: Q값을 얼마나 빠르게 갱신할지 (0~1)
gamma         = 0.9    # 할인율: 미래 보상을 현재 가치로 얼마나 반영할지 (0~1)
epsilon       = 1.0    # 탐험율: 1.0 = 완전 탐험, 0.1 = 거의 이용
epsilon_decay = 0.995  # 매 에피소드마다 epsilon에 곱해지는 감소율
epsilon_min   = 0.1    # epsilon의 최솟값 (이 이하로 내려가지 않음)

episodes  = 1000  # 총 학습 에피소드 수
max_steps = 30    # 에피소드 하나에서 최대 이동 횟수

In [6]:
# 환경 이동 함수 (step)
def step(pos, action):
    row, col = pos
    dr, dc = actions[action]  # 행동에 따른 행·열 변화량

    next_row = row + dr
    next_col = col + dc

    # 경계 밖으로 나가면 제자리 유지, 패널티 -2
    if next_row < 0 or next_row >= ROWS or next_col < 0 or next_col >= COLS:
        return pos, -2, False

    next_pos = (next_row, next_col)

    if next_pos == GOAL:
        return next_pos, +10, True   # 목표 도달: 보상 +10, 에피소드 종료
    elif next_pos == TRAP:
        return next_pos, -10, True   # 함정 도달: 보상 -10, 에피소드 종료
    else:
        return next_pos, -1, False   # 일반 이동: 보상 -1 (빠른 경로 유도)

In [7]:
# epsilon-greedy 행동 선택
def choose_action(state, epsilon):
    if random.random() < epsilon:
        # 탐험: 무작위 행동 선택 (새로운 경험 수집)
        return random.randint(0, num_actions - 1)
    else:
        # 이용: Q-table에서 가장 높은 값의 행동 선택
        return np.argmax(Q[state])

In [8]:
# Q-learning 학습 루프
for episode in range(episodes):
    pos   = START
    state = pos_to_state(pos)  # 시작 상태: (0,0) → 0

    for step_count in range(max_steps):
        # 1. 행동 선택 (epsilon-greedy)
        action = choose_action(state, epsilon)

        # 2. 행동 실행 → 결과 반환
        next_pos, reward, done = step(pos, action)
        next_state = pos_to_state(next_pos)

        # 3. Q-table 업데이트 (벨만 방정식)
        old_q  = Q[state, action]
        # Q(s, a) = r + γ × max Q(s', a')
        target   = reward + gamma * np.max(Q[next_state])
        td_error = target - old_q                      # 목표값과 현재 Q값의 차이
        Q[state][action] = old_q + alpha * td_error    # Q값 갱신

        # 4. 상태 이동
        pos   = next_pos
        state = next_state

        if done:
            break

    # 에피소드 종료 후 epsilon 감소 (탐험 → 이용으로 전환)
    epsilon = max(epsilon_min, epsilon_decay * epsilon)

In [9]:
# 학습된 Q-table 출력
print(np.round(Q, 2))

# 각 상태에서 최선의 행동 출력
for state in range(num_states):
    pos = state_to_pos(state)
    if pos == GOAL:
        print(f'상태 {state} {pos} : 목표지점')
        continue
    if pos == TRAP:
        print(f'상태 {state} {pos} : 함정')  # Q값이 전부 0 (학습 불가)
        continue
    best_action = np.argmax(Q[state])
    print(f'상태 {state} {pos} : {action_names[best_action]}')

[[ 0.77  1.83  0.77  3.12]
 [ 2.05 -9.99  1.68  4.58]
 [ 3.48  5.99  2.98  6.2 ]
 [ 5.17  8.    4.36  5.13]
 [-0.59  3.85 -2.31 -9.28]
 [ 0.    0.    0.    0.  ]
 [ 1.91  7.99 -7.46  4.58]
 [ 6.01 10.    5.79  6.89]
 [-0.75 -1.52 -1.24  5.93]
 [-6.51  0.4   0.43  7.94]
 [ 3.72  3.36  2.02 10.  ]
 [ 0.    0.    0.    0.  ]]
상태 0 (0, 0) : 오른쪽
상태 1 (0, 1) : 오른쪽
상태 2 (0, 2) : 오른쪽
상태 3 (0, 3) : 아래
상태 4 (1, 0) : 아래
상태 5 (1, 1) : 함정
상태 6 (1, 2) : 아래
상태 7 (1, 3) : 아래
상태 8 (2, 0) : 오른쪽
상태 9 (2, 1) : 오른쪽
상태 10 (2, 2) : 오른쪽
상태 11 (2, 3) : 목표지점


In [10]:
# 학습된 Q-table로 실제 경로 추적
pos  = START
path = [pos]

for i in range(20):
    state  = pos_to_state(pos)
    action = np.argmax(Q[state])  # 탐험 없이 최선의 행동만 선택

    next_pos, reward, done = step(pos, action)
    path.append(next_pos)
    pos = next_pos

    if done:
        break

print('이동 경로:', path)
# 결과: [(0,0), (0,1), (0,2), (1,2), (1,3), (2,3)]

이동 경로: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
